# RankMixer 全量验证复评

这个 notebook 只做 full validation evaluation：加载 `outputs/*_best_state.pt`，在完整 `val_user_item_feats_df_all.csv` 上重新计算指标，不重新训练模型。

In [1]:
from __future__ import annotations

import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from rec.config import ExperimentConfig
from rec.data import load_feature_frames, make_loader, merge_history_frames, move_batch_to_device, prepare_ranking_data
from rec.evaluate import RankingEvaluator
from rec.features import build_feature_preset
from rec.models.rankmixer import count_parameters
from rec.pipeline import DEFAULT_SCENARIOS, build_model
from rec.train import default_device

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

CONFIG = ExperimentConfig()
FULL_EVAL_SCENARIOS = ["old_ge", "query_softmax_ce", "listwise_bpr_bce"]
DEVICE = default_device(CONFIG.train.device)

print(f"project_root={CONFIG.paths.project_root}")
print(f"save_path={CONFIG.paths.save_path}")
print(f"output_path={CONFIG.paths.output_path}")
print(f"device={DEVICE}")
print(f"scenarios={FULL_EVAL_SCENARIOS}")

project_root=/Users/lixiang/Developer/funrec-new-rec
save_path=/Users/lixiang/Developer/funrec-new-rec/data/processed/temp_results
output_path=/Users/lixiang/Developer/funrec-new-rec/outputs
device=mps
chunk_size=200000
scenarios=['old_ge', 'query_softmax_ce', 'listwise_bpr_bce']


## 1. 加载完整验证集并准备模型输入

这里不调用 `sample_validation_frame`，所以 `prepared_data.val_frame` 是完整验证集。稀疏词表和 dense scaler 仍然只从训练集拟合，和训练阶段保持一致。

In [2]:
started_at = time.time()

preset = build_feature_preset(
    name="full_val_eval",
    article_svd_dim=CONFIG.data.article_svd_dim,
)

loaded_frames = load_feature_frames(CONFIG.paths, CONFIG.data)
loaded_shape_df = pd.DataFrame([
    {"split": "train", "shape": loaded_frames.train.shape},
    {"split": "val", "shape": None if loaded_frames.val is None else loaded_frames.val.shape},
    {"split": "test", "shape": loaded_frames.test.shape},
    {"split": "history", "shape": loaded_frames.history.shape},
])
display(loaded_shape_df)

merged_frames = merge_history_frames(loaded_frames)
prepared_data = prepare_ranking_data(merged_frames, preset, CONFIG.data)

prepared_shape_df = pd.DataFrame([
    {"name": "train_frame", "shape": prepared_data.train_frame.shape},
    {"name": "val_frame", "shape": None if prepared_data.val_frame is None else prepared_data.val_frame.shape},
    {"name": "val_hit_frame", "shape": None if prepared_data.val_hit_frame is None else prepared_data.val_hit_frame.shape},
    {"name": "test_frame", "shape": prepared_data.test_frame.shape},
    {"name": "x_train_rows", "shape": len(prepared_data.y_train)},
    {"name": "x_val_rows", "shape": None if prepared_data.y_val is None else len(prepared_data.y_val)},
])
display(prepared_shape_df)

print(f"prepare_elapsed_sec={time.time() - started_at:.1f}")

name,value
train_rows_streamed,7024640.0
train_users,109760.0
val_rows_file,8000000.0
sparse_vocab_click_article_id,21545.0
prepare_elapsed_sec,21.7


## 2. 定义全量评估函数

每个 scenario 都会加载对应的 `best_state.pt`，批量预测完整 val，然后保存 full-val summary 和 prediction 明细。

In [3]:
def clear_eval_cache() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if hasattr(torch, "mps") and hasattr(torch.mps, "empty_cache"):
        try:
            torch.mps.empty_cache()
        except Exception:
            pass


def raw_predict_outputs_with_progress(model: torch.nn.Module, x_data: dict[str, np.ndarray], batch_size: int, device: torch.device) -> np.ndarray:
    loader = make_loader(x_data, batch_size=batch_size)
    total_batches = len(loader)
    outputs: list[np.ndarray] = []
    model.eval()
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader, start=1):
            batch = move_batch_to_device(batch, device)
            outputs.append(model(batch).detach().cpu().numpy())
            if batch_idx == 1 or batch_idx % 200 == 0 or batch_idx == total_batches:
                print(f"predict batch {batch_idx}/{total_batches}", flush=True)
    return np.concatenate(outputs, axis=0)


def read_sampled_summary(scenario_name: str) -> dict[str, object]:
    path = CONFIG.paths.output_path / f"{scenario_name}_summary.csv"
    if not path.exists():
        return {}
    row = pd.read_csv(path).tail(1).iloc[0].to_dict()
    return {f"sampled_{key}": value for key, value in row.items() if key in {"best_epoch", "best_metric", "full_mrr", "hit_mrr"}}


def evaluate_full_val(scenario_name: str) -> tuple[pd.DataFrame, dict[str, Path]]:
    if prepared_data.x_val is None or prepared_data.y_val is None or prepared_data.val_frame is None:
        raise ValueError("prepared_data does not contain validation data")

    scenario = DEFAULT_SCENARIOS[scenario_name]
    state_path = CONFIG.paths.output_path / f"{scenario_name}_best_state.pt"
    if not state_path.exists():
        raise FileNotFoundError(state_path)

    print(f"\n=== full val eval: {scenario_name} ===", flush=True)
    print(f"protocol={scenario.loss_name} + {scenario.head_type} + {scenario.score_mode}", flush=True)
    print(f"state_path={state_path}", flush=True)

    model = build_model(scenario, prepared_data, CONFIG.model, CONFIG.data.article_svd_dim).to(DEVICE)
    print(f"params={count_parameters(model):,}", flush=True)
    model.load_state_dict(torch.load(state_path, map_location="cpu"))

    scenario_started_at = time.time()
    raw_output = raw_predict_outputs_with_progress(model, prepared_data.x_val, CONFIG.train.batch_size, DEVICE)
    evaluator = RankingEvaluator(topk=CONFIG.train.topk)
    eval_result = evaluator.evaluate_raw(
        raw_output,
        prepared_data.y_val,
        prepared_data.val_frame,
        head_type=scenario.head_type,
        score_mode=scenario.score_mode,
        hit_mask=prepared_data.val_hit_mask,
    )

    row = {
        "scenario": scenario_name,
        "protocol": f"{scenario.loss_name} + {scenario.head_type} + {scenario.score_mode}",
        "eval_scope": "full_validation",
        "val_user_sample_size": None,
        "full_mrr": eval_result.get("full_mrr"),
        "full_ndcg": eval_result.get("full_ndcg"),
        "full_hit_rate": eval_result.get("full_hit_rate"),
        "full_query_count": eval_result.get("full_query_count"),
        "full_pos_query_count": eval_result.get("full_pos_query_count"),
        "hit_mrr": eval_result.get("hit_mrr"),
        "hit_ndcg": eval_result.get("hit_ndcg"),
        "hit_hit_rate": eval_result.get("hit_hit_rate"),
        "hit_query_count": eval_result.get("hit_query_count"),
        "hit_pos_query_count": eval_result.get("hit_pos_query_count"),
        "elapsed_sec": time.time() - scenario_started_at,
    }
    row.update(read_sampled_summary(scenario_name))

    summary_df = pd.DataFrame([row])
    summary_path = CONFIG.paths.output_path / f"{scenario_name}_full_val_summary.csv"
    pred_path = CONFIG.paths.output_path / f"{scenario_name}_full_val_predictions.csv"
    hit_pred_path = CONFIG.paths.output_path / f"{scenario_name}_full_val_hit_predictions.csv"

    summary_df.to_csv(summary_path, index=False)
    eval_result["val_pred_df"].to_csv(pred_path, index=False)
    artifacts = {"summary": summary_path, "predictions": pred_path}
    if eval_result.get("hit_pred_df") is not None:
        eval_result["hit_pred_df"].to_csv(hit_pred_path, index=False)
        artifacts["hit_predictions"] = hit_pred_path

    display(summary_df)
    display(pd.DataFrame({"artifact": list(artifacts), "path": [str(path) for path in artifacts.values()]}))

    del model, raw_output, eval_result
    clear_eval_cache()
    return summary_df, artifacts

streaming full-val helper functions loaded


## 3. 跑三组全量验证

这个单元格可能会运行比较久。每 200 个 prediction batch 会打印一次进度。

In [4]:
all_summary_frames = []
all_artifact_rows = []

for scenario_name in FULL_EVAL_SCENARIOS:
    summary_df, artifacts = evaluate_full_val(scenario_name)
    all_summary_frames.append(summary_df)
    for artifact, path in artifacts.items():
        all_artifact_rows.append({"scenario": scenario_name, "artifact": artifact, "path": str(path)})

full_val_summary = pd.concat(all_summary_frames, ignore_index=True)
full_val_artifacts = pd.DataFrame(all_artifact_rows)

combined_summary_path = CONFIG.paths.output_path / "rankmixerclean_full_val_summary.csv"
combined_artifacts_path = CONFIG.paths.output_path / "rankmixerclean_full_val_artifacts.csv"
full_val_summary.to_csv(combined_summary_path, index=False)
full_val_artifacts.to_csv(combined_artifacts_path, index=False)

display(full_val_summary)
display(full_val_artifacts)
print(f"combined_summary={combined_summary_path}")
print(f"combined_artifacts={combined_artifacts_path}")

scenario,protocol,eval_scope,val_user_sample_size,elapsed_sec,full_auc,full_mrr,full_ndcg,full_hit_rate,full_query_count,full_pos_query_count,hit_auc,hit_mrr,hit_ndcg,hit_hit_rate,hit_query_count,hit_pos_query_count,sampled_best_epoch,sampled_best_metric,sampled_full_mrr,sampled_hit_mrr
old_ge,old_ge + two_logit + diff,full_validation_streaming,None,1850.642188,0.935124,0.016557,0.017131,0.018818,1424843,27382,0.927461,0.86155,0.891431,0.979183,27382,27382,5,0.205317,0.205317,0.305077


artifact,path
summary,/Users/lixiang/Developer/funrec-new-rec/outputs/old_ge_full_val_summary.csv
predictions,/Users/lixiang/Developer/funrec-new-rec/outputs/old_ge_full_val_predictions.csv
hit_predictions,/Users/lixiang/Developer/funrec-new-rec/outputs/old_ge_full_val_hit_predictions.csv


## 4. 只看核心指标

最后看这个表即可：`sampled_*` 是之前 2000 用户抽样验证的结果，`full_*` 是本 notebook 跑出的全量验证结果。

In [ ]:
metric_cols = [
    "scenario",
    "protocol",
    "sampled_best_epoch",
    "sampled_full_mrr",
    "full_mrr",
    "sampled_hit_mrr",
    "hit_mrr",
    "full_ndcg",
    "hit_ndcg",
    "elapsed_sec",
]
display(full_val_summary[[col for col in metric_cols if col in full_val_summary.columns]])